# Depression Detection from Video Features using BiLSTM with Attention (DAIC-WOZ Dataset)

This notebook implements a complete machine learning pipeline for detecting depression from video-derived features, utilizing the DAIC-WOZ dataset. The pipeline encompasses data loading, extensive preprocessing, feature engineering through windowing, data standardization, and training a deep learning model based on Bidirectional Long Short-Term Memory (BiLSTM) networks with an attention mechanism for both binary classification (depressed/not depressed) and regression (PHQ-8 score prediction). The model is thoroughly evaluated using standard classification and regression metrics.

**Key Steps Covered:**

1.  **Data Loading and Initial Exploration**: Loading raw OpenFace features and PHQ-8 labels, and examining the data structure.
2.  **Data Preprocessing**: Implementing functions to filter frames based on `confidence` and `success` scores, and selecting relevant features by dropping unnecessary columns.
3.  **Data Windowing**: Creating overlapping time windows from the preprocessed time-series data to capture temporal dependencies.
4.  **Metadata Generation and Data Splitting**: Preparing a metadata DataFrame that links participant information, labels, and window files, and splitting the data into training, validation, and test sets based on pre-defined participant splits.
5.  **Feature Scaling**: Standardizing features using `StandardScaler` fitted only on the training data to prevent data leakage.
6.  **PyTorch Dataset and DataLoader**: Implementing custom `EDAICDataset` and `DataLoader` classes for efficient batch processing of the windowed data in PyTorch.
7.  **Model Architecture**: Defining `DepressionNet`, a BiLSTM model with an attention pooling layer, followed by shared features and separate heads for binary classification and PHQ-8 regression.
8.  **Training and Validation**: Setting up loss functions (CrossEntropyLoss with class weights for classification and MSELoss for regression), an optimizer (AdamW), and implementing training and validation loops with early stopping based on F1-score.
9.  **Model Evaluation**: Evaluating the final model on the unseen test set using a comprehensive set of metrics including accuracy, precision, recall, F1-score, confusion matrix for classification, and MAE, RMSE, R2 for regression.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive")

# Change this after checking your Drive structure
PROCESSED_DIR = BASE_DIR / "DAIC_WOZ_VIDEO_BRANCH" / "data" / "processed"
RAW_DATA = BASE_DIR / "edaic3.0" / "313" / "features"

print("Exists:", PROCESSED_DIR.exists())
print("Path:", PROCESSED_DIR)

Exists: True
Path: /content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/data/processed


In [ ]:
df = pd.read_csv(RAW_DATA/"313_OpenFace2.1.0_Pose_gaze_AUs.csv")

In [ ]:
df

,frame,timestamp,confidence,success,pose_Tx,pose_Ty,pose_Tz,pose_Rx,pose_Ry,pose_Rz,...,AU12_c,AU14_c,AU15_c,AU17_c,AU20_c,AU23_c,AU25_c,AU26_c,AU28_c,AU45_c
0,1,0.000,0.98,1,15.3,13.2,546.1,0.285,-0.024,-0.031,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.033,0.98,1,15.2,12.9,544.4,0.304,-0.025,-0.031,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.066,0.98,1,15.3,12.8,543.5,0.317,-0.022,-0.035,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.099,0.98,1,15.3,12.8,543.3,0.317,-0.022,-0.035,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.132,0.98,1,15.3,12.8,543.3,0.317,-0.022,-0.035,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22843,22844,753.646,0.98,1,52.2,34.6,551.2,0.352,0.098,0.090,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
22844,22845,753.679,0.98,1,52.5,34.8,550.1,0.356,0.097,0.089,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
22845,22846,753.712,0.98,1,52.7,35.1,550.5,0.354,0.091,0.090,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
22846,22847,753.745,0.98,1,53.1,35.2,550.0,0.354,0.085,0.088,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0


In [ ]:
df.shape

(22848, 53)

In [ ]:
print(df.columns.tolist())

['frame', 'timestamp', 'confidence', 'success', 'pose_Tx', 'pose_Ty', 'pose_Tz', 'pose_Rx', 'pose_Ry', 'pose_Rz', 'gaze_0_x', 'gaze_0_y', 'gaze_0_z', 'gaze_1_x', 'gaze_1_y', 'gaze_1_z', 'gaze_angle_x', 'gaze_angle_y', 'AU01_r', 'AU02_r', 'AU04_r', 'AU05_r', 'AU06_r', 'AU07_r', 'AU09_r', 'AU10_r', 'AU12_r', 'AU14_r', 'AU15_r', 'AU17_r', 'AU20_r', 'AU23_r', 'AU25_r', 'AU26_r', 'AU45_r', 'AU01_c', 'AU02_c', 'AU04_c', 'AU05_c', 'AU06_c', 'AU07_c', 'AU09_c', 'AU10_c', 'AU12_c', 'AU14_c', 'AU15_c', 'AU17_c', 'AU20_c', 'AU23_c', 'AU25_c', 'AU26_c', 'AU28_c', 'AU45_c']


In [ ]:
def load_participant(path):

  df = pd.read_csv(path)
  print(f"load from path: {path} sucessful!")
  return df

def filter_frame(df):
  df_new = pd.DataFrame()
  df_filtered = df[
    (df["success"] == 1) &
    (df["confidence"] >= 0.8)
]
  original_frames = len(df)
  filtered_frames = len(df_filtered)

  print(f"Original: {original_frames}")
  print(f"Filtered: {filtered_frames}")
  print(f"Removed: {(original_frames-filtered_frames)/original_frames:.2%}")
  return df_filtered


def select_feature(df,drop_features):
  df=df.drop(columns=drop_features)
  return df

def save_processed_file(df,id):
  df.to_csv(PROCESSED_DIR / "new_processed" / f"{id}_nprocessed.csv", index=False)


In [ ]:
label_df = pd.read_csv(BASE_DIR / "edaic3.0" / "labels" / "Detailed_PHQ8_Labels.csv")
session_ids = np.array(label_df["Participant_ID"])
session_ids

array([300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312,
       313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325,
       326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338,
       339, 340, 341, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352,
       353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365,
       366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378,
       379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391,
       392, 393, 395, 396, 397, 399, 400, 401, 402, 403, 404, 405, 406,
       407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419,
       420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432,
       433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445,
       446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458,
       459, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472,
       473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 48

In [ ]:
label_df.head()

,Participant_ID,PHQ_8NoInterest,PHQ_8Depressed,PHQ_8Sleep,PHQ_8Tired,PHQ_8Appetite,PHQ_8Failure,PHQ_8Concentrating,PHQ_8Moving,PHQ_8Total,PHQ8_Binary,PHQ8_MultiClass
0,300,0,0,1,0,1,0,0,0,2,0,No Depression
1,301,0,0,1,1,1,0,0,0,3,0,No Depression
2,302,1,1,0,1,0,1,0,0,4,0,No Depression
3,303,0,0,0,0,0,0,0,0,0,0,No Depression
4,304,0,1,1,2,2,0,0,0,6,0,No Depression


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ---------------------------
# CONFIG
# ---------------------------

BASE_DIR = Path("/content/drive/MyDrive")

drop_features = [
    "frame",
    "timestamp",
    "confidence",
    "success"
]

# ---------------------------
# DRIVER
# ---------------------------

def preprocessing_driver_script(session_ids, drop_features):

    summary = []

    for participant_id in session_ids:

        try:

            print(f"\nProcessing Participant {participant_id}...")

            file_path = (
                BASE_DIR
                / "edaic3.0"
                / str(participant_id)
                / "features"
                / f"{participant_id}_OpenFace2.1.0_Pose_gaze_AUs.csv"
            )

            # Load
            df = load_participant(file_path)

            original_frames = len(df)

            # Filter
            df_filtered = filter_frame(df)

            filtered_frames = len(df_filtered)

            removal_ratio = (
                (original_frames - filtered_frames)
                / original_frames
            )

            # Feature selection
            df_filtered = select_feature(
                df_filtered,
                drop_features
            )

            # Save
            save_processed_file(
                df_filtered,
                participant_id
            )

            # Summary row
            summary.append({
                "participant_id": participant_id,
                "original_frames": original_frames,
                "filtered_frames": filtered_frames,
                "removal_ratio": removal_ratio,
                "num_features": df_filtered.shape[1]
            })

            print(
                f"Saved | Frames: "
                f"{original_frames} -> {filtered_frames}"
            )

        except Exception as e:

            print(
                f"Error processing "
                f"{participant_id}: {e}"
            )

    summary_df = pd.DataFrame(summary)

    summary_df.to_csv(
        PROCESSED_DIR
        / "new_processed"
        / "preprocessing_summary.csv",
        index=False
    )

    return summary_df

In [ ]:
label_df = pd.read_csv(
    BASE_DIR
    / "edaic3.0"
    / "labels"
    / "Detailed_PHQ8_Labels.csv"
)

session_ids = label_df["Participant_ID"].tolist()

summary_df = preprocessing_driver_script(
    session_ids,   # test first
    drop_features
)


Processing Participant 300...
load from path: /content/drive/MyDrive/edaic3.0/300/features/300_OpenFace2.1.0_Pose_gaze_AUs.csv sucessful!
Original: 19458
Filtered: 19378
Removed: 0.41%
Saved | Frames: 19458 -> 19378

Processing Participant 301...
load from path: /content/drive/MyDrive/edaic3.0/301/features/301_OpenFace2.1.0_Pose_gaze_AUs.csv sucessful!
Original: 24721
Filtered: 24618
Removed: 0.42%
Saved | Frames: 24721 -> 24618

Processing Participant 302...
load from path: /content/drive/MyDrive/edaic3.0/302/features/302_OpenFace2.1.0_Pose_gaze_AUs.csv sucessful!
Original: 22766
Filtered: 22130
Removed: 2.79%
Saved | Frames: 22766 -> 22130

Processing Participant 303...
load from path: /content/drive/MyDrive/edaic3.0/303/features/303_OpenFace2.1.0_Pose_gaze_AUs.csv sucessful!
Original: 29565
Filtered: 29400
Removed: 0.56%
Saved | Frames: 29565 -> 29400

Processing Participant 304...
load from path: /content/drive/MyDrive/edaic3.0/304/features/304_OpenFace2.1.0_Pose_gaze_AUs.csv suce

In [ ]:
summary_df.head()

,participant_id,original_frames,filtered_frames,removal_ratio,num_features
0,300,19458,19378,0.004111,49
1,301,24721,24618,0.004166,49
2,302,22766,22130,0.027936,49
3,303,29565,29400,0.005581,49
4,304,23780,23368,0.017325,49


In [ ]:
len(summary_df)

219

In [ ]:
FPS = 30

WINDOW_SIZE = 300   # 10 sec
STRIDE = 150        # 5 sec overlap

In [ ]:
import numpy as np

def generate_windows(df, window_size=300, stride=150):
    data = df.values
    windows = []
    start = 0
    while start + window_size <= len(data):
        window = data[start:start + window_size]
        windows.append(window)
        start += stride
    windows = np.array(windows)
    return windows



In [ ]:
df = pd.read_csv(
    PROCESSED_DIR /"new_processed"/ "300_nprocessed.csv"
)

windows = generate_windows(
    df,
    window_size=300,
    stride=150
)

print(windows.shape)

(128, 300, 49)


In [ ]:
print(windows[0].shape)

(300, 49)


In [ ]:
print(windows[0][:5])

[[ 6.95000e+01  3.77000e+01  5.76800e+02  2.21000e-01  3.60000e-02
  -6.80000e-02  1.53240e-02  2.98824e-01 -9.54185e-01 -1.87119e-01
   3.35262e-01 -9.23356e-01 -9.10000e-02  3.26000e-01  0.00000e+00
   0.00000e+00  5.90000e-01  5.30000e-01  0.00000e+00  0.00000e+00
   0.00000e+00  1.60000e-01  0.00000e+00  2.12000e+00  4.00000e-02
   5.10000e-01  0.00000e+00  1.40000e+00  1.50000e-01  6.40000e-01
   4.10000e-01  0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00
   0.00000e+00  0.00000e+00  0.00000e+00  0.00000e+00  0.00000e+00
   1.00000e+00  0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00
   0.00000e+00  0.00000e+00  0.00000e+00  0.00000e+00]
 [ 6.94000e+01  3.74000e+01  5.77500e+02  2.19000e-01  3.70000e-02
  -6.70000e-02  6.17400e-03  2.94520e-01 -9.55626e-01 -1.91460e-01
   3.36481e-01 -9.22021e-01 -9.80000e-02  3.24000e-01  0.00000e+00
   0.00000e+00  9.10000e-01  4.30000e-01  0.00000e+00  0.00000e+00
   0.00000e+00  1.90000e-01  0.00000e+00  2.08000e+00  1.00000e-02
   1.14

In [ ]:
WINDOWS_DIR = PROCESSED_DIR / "windows_processed"

In [ ]:
def save_windows(windows, participant_id):

    save_path = (
        WINDOWS_DIR /
        f"{participant_id}_windows.npy"
    )

    np.save(save_path, windows)

In [ ]:
for participant_id in session_ids:

    df = pd.read_csv(
        PROCESSED_DIR / "new_processed"/
        f"{participant_id}_nprocessed.csv"
    )

    windows = generate_windows(
        df,
        window_size=300,
        stride=150
    )

    save_windows(
        windows,
        participant_id
    )

    print(
        f"{participant_id}: "
        f"{windows.shape[0]} windows"
    )

300: 128 windows
301: 163 windows
302: 146 windows
303: 195 windows
304: 154 windows
305: 338 windows
306: 169 windows
307: 244 windows
308: 167 windows
309: 140 windows
310: 167 windows
311: 155 windows
312: 157 windows
313: 149 windows
314: 309 windows
315: 192 windows
316: 173 windows
317: 161 windows
318: 116 windows
319: 132 windows
320: 166 windows
321: 162 windows
322: 207 windows
323: 161 windows
324: 141 windows
325: 173 windows
326: 136 windows
327: 134 windows
328: 210 windows
329: 138 windows
330: 151 windows
331: 167 windows
332: 173 windows
333: 192 windows
334: 193 windows
335: 163 windows
336: 185 windows
337: 372 windows
338: 114 windows
339: 170 windows
340: 118 windows
341: 172 windows
343: 175 windows
344: 215 windows
345: 156 windows
346: 242 windows
347: 119 windows
348: 142 windows
349: 239 windows
350: 174 windows
351: 151 windows
352: 150 windows
353: 155 windows
354: 113 windows
355: 131 windows
356: 187 windows
357: 78 windows
358: 121 windows
359: 199 window

In [ ]:
window_counts = []

for participant_id in session_ids:

    df = pd.read_csv(
        PROCESSED_DIR / "new_processed"/
        f"{participant_id}_nprocessed.csv"
    )

    windows = generate_windows(df)

    window_counts.append(
        windows.shape[0]
    )

print("Min:", min(window_counts))
print("Max:", max(window_counts))
print("Mean:", np.mean(window_counts))
print("Total:", len(window_counts))

Min: 78
Max: 389
Mean: 188.70319634703196
Total: 219


In [ ]:
CSV_PATH = "/content/drive/MyDrive/DAIC_WOZ_AUDIO_BRANCH/data/processed/wav2vec_5utt_segments_participant_split.csv"

df_split = pd.read_csv(CSV_PATH)
df_split = df_split.drop(columns=["sample_id", "sample_path", "num_samples", "duration_sec", "binary_label", "PHQ8_Score", "num_segments_merged"])
df_split.loc[438] = [438, "train"] # Corrected to assign both Participant_ID and split
df_unique = df_split.drop_duplicates(subset=["Participant_ID"], keep="first")
df_unique.head()

,Participant_ID,split
0,300,train
15,301,test
22,302,val
41,303,test
50,304,train


In [ ]:
# Find IDs in df1 that are NOT in df2
missing_in_df2 = set(label_df["Participant_ID"]) - set(df_unique["Participant_ID"])
print("Missing in df2:", missing_in_df2)

# Find IDs in df2 that are NOT in df1
missing_in_df1 = set(df_unique["Participant_ID"]) - set(label_df["Participant_ID"])
print("Missing in df1:", missing_in_df1)

Missing in df2: set()
Missing in df1: set()


In [ ]:
df_unique.shape

(219, 2)

In [ ]:
label_df = pd.merge(df_unique, label_df, on="Participant_ID", how="inner")

label_df.head()
label_df.shape

(219, 13)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def build_metadata(label_df, windows_dir):

    metadata = []

    for _, row in label_df.iterrows():

        participant_id = row["Participant_ID"]

        window_file = windows_dir / f"{participant_id}_windows.npy"

        if not window_file.exists():

            print(f"Missing windows for {participant_id}")
            continue

        windows = np.load(window_file)

        metadata.append({
            "participant_id": participant_id,
            "phq_score": row["PHQ_8Total"],
            "binary_label": row["PHQ8_Binary"],
            "num_windows": windows.shape[0],
            "window_file": str(window_file),
            "split":row["split"]
        })

    metadata_df = pd.DataFrame(metadata)

    return metadata_df

In [ ]:
metadata_df = build_metadata(
    label_df,
    WINDOWS_DIR
)

metadata_df.head()

,participant_id,phq_score,binary_label,num_windows,window_file,split
0,300,2,0,128,/content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/d...,train
1,301,3,0,163,/content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/d...,test
2,302,4,0,146,/content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/d...,val
3,303,0,0,195,/content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/d...,test
4,304,6,0,154,/content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/d...,train


In [ ]:
metadata_df.to_csv(
    # BASE_DIR / "edaic3.0" / "metadata.csv",
    WINDOWS_DIR/"metadata.csv",
    index=False
)

In [ ]:
import pandas as pd
import numpy as np

metadata_df = pd.read_csv(
    WINDOWS_DIR/"metadata.csv"
)

In [ ]:
train_df = metadata_df[
    metadata_df["split"] == "train"
]

In [ ]:
train_df.shape

(131, 6)

In [ ]:
all_train_frames = []

for _, row in train_df.iterrows():

    windows = np.load(
        row["window_file"]
    )

    frames = windows.reshape(
        -1,
        windows.shape[-1]
    )

    all_train_frames.append(frames)

In [ ]:
all_train_frames = np.concatenate(
    all_train_frames,
    axis=0
)

print(all_train_frames.shape)

(7309500, 49)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(all_train_frames)

StandardScaler()

In [ ]:
import joblib

joblib.dump(
    scaler,
    PROCESSED_DIR/ "scaler.pkl"
)

['/content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/data/processed/scaler.pkl']

In [ ]:
print(scaler.mean_.shape)
print(scaler.scale_.shape)

(49,)
(49,)


In [ ]:
sample = all_train_frames[:1000]

normalized = scaler.transform(sample)

print(normalized.mean(axis=0)[:5])
print(normalized.std(axis=0)[:5])

[ 0.31187817 -0.46481905  0.59503978 -0.28621905 -1.08553923]
[0.75634686 0.62155709 0.3345675  1.21278714 1.45562954]


In [ ]:
metadata_df["window_file"][0]

'/content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/data/processed/windows_processed/300_windows.npy'

In [ ]:
import torch

In [ ]:
from torch.utils.data import Dataset

class EDAICDataset(Dataset):

    def __init__(self, metadata_df):

        self.samples = []

        # iterate over metadata rows
        for _,row in metadata_df.iterrows():

        # load num_windows
          for win_idx in range(row["num_windows"]):

        # create one sample entry
            sample = {
                 "participant_id": row["participant_id"],
                  "window_idx": win_idx,
                  "phq_score": row["phq_score"],
                  "binary_label": row["binary_label"],
                  "window_file": row["window_file"]
            }
            self.samples.append(sample)

        # for each window

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        sample = self.samples[idx]

        # load npy
        windows = np.load(sample["window_file"])

        # extract window_idx
        window = windows[sample["window_idx"]]

        # return tensors
        binary_label = sample["binary_label"]
        phq_score = sample["phq_score"]
        participant_id = sample["participant_id"]
        window = torch.tensor(
            window,
            dtype=torch.float32
        )

        binary_label = torch.tensor(
            binary_label,
            dtype=torch.long
        )

        phq_score = torch.tensor(
            phq_score,
            dtype=torch.float32
        )
        return {
            "window": window,
            "binary_label": binary_label,
            "phq_score": phq_score,
            "participant_id": participant_id
        }

In [ ]:
dataset = EDAICDataset(metadata_df)

sample = dataset[0]

print(sample["window"].shape)
print(sample["binary_label"])
print(sample["phq_score"])
print(sample["participant_id"])

torch.Size([300, 49])
tensor(0)
tensor(2.)
300


In [ ]:
from torch.utils.data import DataLoader

train_df = metadata_df[
    metadata_df["split"] == "train"
]

val_df = metadata_df[
    metadata_df["split"] == "val"
]

test_df = metadata_df[
    metadata_df["split"] == "test"
]

In [ ]:
train_dataset = EDAICDataset(train_df)
val_dataset = EDAICDataset(val_df)
test_dataset = EDAICDataset(test_df)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
batch = next(iter(train_loader))

print(batch["window"].shape)
print(batch["binary_label"].shape)
print(batch["phq_score"].shape)

torch.Size([32, 300, 49])
torch.Size([32])
torch.Size([32])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionPool(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        """
        x: (B, T, H)
        """

        scores = self.attention(x).squeeze(-1)   # (B, T)

        weights = torch.softmax(scores, dim=1)   # (B, T)

        context = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )                                         # (B, H)

        return context, weights


class DepressionNet(nn.Module):

    def __init__(
        self,
        input_size=49,
        hidden_size=128,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        lstm_output_dim = hidden_size * 2  # 256

        # Attention
        self.attention = AttentionPool(
            lstm_output_dim
        )

        # Shared representation
        self.shared = nn.Sequential(
            nn.Linear(lstm_output_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

        # Regression head
        self.regressor = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        """
        x: (B,300,49)
        """

        lstm_out, _ = self.lstm(x)
        # (B,300,256)

        context, attention_weights = self.attention(
            lstm_out
        )
        # (B,256)

        shared_features = self.shared(
            context
        )
        # (B,128)

        class_logits = self.classifier(
            shared_features
        )
        # (B,2)

        phq_score = self.regressor(
            shared_features
        )
        # (B,1)

        return {
            "class_logits": class_logits,
            "phq_score": phq_score,
            "attention": attention_weights
        }

In [ ]:
model = DepressionNet()

x = torch.randn(
    32,
    300,
    49
)

outputs = model(x)

print(outputs["class_logits"].shape)
print(outputs["phq_score"].shape)
print(outputs["attention"].shape)

torch.Size([32, 2])
torch.Size([32, 1])
torch.Size([32, 300])


In [ ]:
train_df = metadata_df[
    metadata_df["split"] == "train"
]

print(train_df["binary_label"].value_counts())

binary_label
0    92
1    39
Name: count, dtype: int64


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

labels = train_df["binary_label"].values

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print(class_weights)

tensor([0.7120, 1.6795])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DepressionNet().to(device)

In [ ]:
classification_criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

regression_criterion = nn.MSELoss()

In [ ]:
batch = next(iter(train_loader))

print(batch["window"].shape)
print(batch["binary_label"].shape)
print(batch["phq_score"].shape)

print(batch["window"].dtype)
print(batch["binary_label"].dtype)
print(batch["phq_score"].dtype)

torch.Size([32, 300, 49])
torch.Size([32])
torch.Size([32])
torch.float32
torch.int64
torch.float32


In [ ]:
def compute_loss(
    outputs,
    binary_labels,
    phq_scores,
    alpha=0.1
):

    classification_loss = classification_criterion(
        outputs["class_logits"],
        binary_labels
    )

    regression_loss = regression_criterion(
        outputs["phq_score"].squeeze(1),
        phq_scores
    )

    total_loss = (
        classification_loss
        + alpha * regression_loss
    )

    return (
        total_loss,
        classification_loss,
        regression_loss
    )

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = DepressionNet().to(device)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    classification_criterion,
    regression_criterion,
    device,
    alpha=0.1
):

    model.train()

    total_loss = 0
    total_cls_loss = 0
    total_reg_loss = 0

    for batch in loader:

        windows = batch["window"].to(device)

        binary_labels = batch["binary_label"].to(device)

        phq_scores = batch["phq_score"].to(device)

        optimizer.zero_grad()

        outputs = model(windows)

        cls_loss = classification_criterion(
            outputs["class_logits"],
            binary_labels
        )

        reg_loss = regression_criterion(
            outputs["phq_score"].squeeze(1),
            phq_scores
        )

        loss = cls_loss + alpha * reg_loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        total_cls_loss += cls_loss.item()
        total_reg_loss += reg_loss.item()

    return (
        total_loss / len(loader),
        total_cls_loss / len(loader),
        total_reg_loss / len(loader)
    )

In [ ]:
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    mean_squared_error
)

import numpy as np

In [ ]:
def validate(
    model,
    loader,
    classification_criterion,
    regression_criterion,
    device,
    alpha=0.1
):

    model.eval()

    total_loss = 0

    all_preds = []
    all_labels = []

    all_phq_preds = []
    all_phq_targets = []

    with torch.no_grad():

        for batch in loader:

            windows = batch["window"].to(device)

            binary_labels = batch["binary_label"].to(device)

            phq_scores = batch["phq_score"].to(device)

            outputs = model(windows)

            cls_loss = classification_criterion(
                outputs["class_logits"],
                binary_labels
            )

            reg_loss = regression_criterion(
                outputs["phq_score"].squeeze(1),
                phq_scores
            )

            loss = cls_loss + alpha * reg_loss

            total_loss += loss.item()

            preds = torch.argmax(
                outputs["class_logits"],
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                binary_labels.cpu().numpy()
            )

            all_phq_preds.extend(
                outputs["phq_score"]
                .squeeze(1)
                .cpu()
                .numpy()
            )

            all_phq_targets.extend(
                phq_scores.cpu().numpy()
            )

    f1 = f1_score(
        all_labels,
        all_preds
    )

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    rmse = np.sqrt(
        mean_squared_error(
            all_phq_targets,
            all_phq_preds
        )
    )

    return (
        total_loss / len(loader),
        acc,
        f1,
        rmse
    )

In [ ]:
num_epochs = 30

best_f1 = 0

print("="*60)
print("Training Started")
print("="*60)
print(f"Device: {device}")
print(f"Train Samples: {len(train_dataset)}")
print(f"Val Samples: {len(val_dataset)}")
print(f"Batch Size: {train_loader.batch_size}")
print(f"Learning Rate: {optimizer.param_groups[0]['lr']}")
print("="*60)

for epoch in range(num_epochs):

    print(f"\n{'='*20} Epoch {epoch+1}/{num_epochs} {'='*20}")

    train_loss, train_cls, train_reg = train_one_epoch(
        model,
        train_loader,
        optimizer,
        classification_criterion,
        regression_criterion,
        device
    )

    val_loss, val_acc, val_f1, val_rmse = validate(
        model,
        val_loader,
        classification_criterion,
        regression_criterion,
        device
    )

    print(f"Train Total Loss : {train_loss:.4f}")
    print(f"Train Cls Loss   : {train_cls:.4f}")
    print(f"Train Reg Loss   : {train_reg:.4f}")

    print(f"Val Loss         : {val_loss:.4f}")
    print(f"Val Accuracy     : {val_acc:.4f}")
    print(f"Val F1 Score     : {val_f1:.4f}")
    print(f"Val RMSE         : {val_rmse:.4f}")

    if val_f1 > best_f1:

        best_f1 = val_f1

        torch.save(
            model.state_dict(),
            "best_model.pt"
        )

        print("\n🔥 New Best Model Saved!")
        print(f"Best F1: {best_f1:.4f}")

    else:

        print(
            f"\nNo Improvement | "
            f"Current Best F1 = {best_f1:.4f}"
        )

    print("-"*60)

Training Started
Device: cuda
Train Samples: 24365
Val Samples: 8197
Batch Size: 32
Learning Rate: 0.001

==================== Epoch 1/30 ====================


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

In [ ]:
def evaluate_model(
    model,
    loader,
    device
):

    model.eval()

    all_cls_preds = []
    all_cls_labels = []

    all_phq_preds = []
    all_phq_labels = []

    with torch.no_grad():

        for batch in loader:

            windows = batch["window"].to(device)

            binary_labels = batch["binary_label"].to(device)

            phq_scores = batch["phq_score"].to(device)

            outputs = model(windows)

            cls_preds = torch.argmax(
                outputs["class_logits"],
                dim=1
            )

            phq_preds = outputs["phq_score"].squeeze(1)

            all_cls_preds.extend(
                cls_preds.cpu().numpy()
            )

            all_cls_labels.extend(
                binary_labels.cpu().numpy()
            )

            all_phq_preds.extend(
                phq_preds.cpu().numpy()
            )

            all_phq_labels.extend(
                phq_scores.cpu().numpy()
            )

    metrics = {

        "accuracy":
            accuracy_score(
                all_cls_labels,
                all_cls_preds
            ),

        "precision":
            precision_score(
                all_cls_labels,
                all_cls_preds,
                zero_division=0
            ),

        "recall":
            recall_score(
                all_cls_labels,
                all_cls_preds,
                zero_division=0
            ),

        "f1":
            f1_score(
                all_cls_labels,
                all_cls_preds,
                zero_division=0
            ),

        "confusion_matrix":
            confusion_matrix(
                all_cls_labels,
                all_cls_preds
            ),

        "mae":
            mean_absolute_error(
                all_phq_labels,
                all_phq_preds
            ),

        "rmse":
            np.sqrt(
                mean_squared_error(
                    all_phq_labels,
                    all_phq_preds
                )
            ),

        "r2":
            r2_score(
                all_phq_labels,
                all_phq_preds
            )
    }

    return metrics

In [ ]:
model.load_state_dict(
    torch.load(
        "best_model.pt",
        map_location=device
    )
)

model.to(device)

In [ ]:
test_metrics = evaluate_model(
    model,
    test_loader,
    device
)

In [ ]:
print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)

print(
    f"Accuracy  : {test_metrics['accuracy']:.4f}"
)

print(
    f"Precision : {test_metrics['precision']:.4f}"
)

print(
    f"Recall    : {test_metrics['recall']:.4f}"
)

print(
    f"F1 Score  : {test_metrics['f1']:.4f}"
)

print(
    f"MAE       : {test_metrics['mae']:.4f}"
)

print(
    f"RMSE      : {test_metrics['rmse']:.4f}"
)

print(
    f"R2 Score  : {test_metrics['r2']:.4f}"
)

print("\nConfusion Matrix:")
print(
    test_metrics["confusion_matrix"]
)